<a href="https://colab.research.google.com/github/yaswanthcs24/AIN-LAB/blob/main/1WN24CS133_Week9_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# First Order Logic - Resolution

# Check whether a term is a variable
def is_variable(x):
    return x in ["X", "Y", "Z", "W", "V", "U", "T"]


# Find the final value of a variable
def substitute(x, subs):
    while x in subs:
        x = subs[x]
    return x


# Unification
def unify(x, y, subs):
    x = substitute(x, subs)
    y = substitute(y, subs)

    if x == y:
        return True

    if is_variable(x):
        subs[x] = y
        return True

    if is_variable(y):
        subs[y] = x
        return True

    return False


# Apply substitutions to a literal
def apply_subs(literal, subs):
    pred, args, neg = literal

    new_args = tuple(substitute(a, subs) for a in args)

    return (pred, new_args, neg)


# Resolution
def resolve(c1, c2):

    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):

            if l1[0] == l2[0] and l1[2] != l2[2]:

                subs = {}

                if len(l1[1]) != len(l2[1]):
                    continue

                valid = True

                for a, b in zip(l1[1], l2[1]):
                    if not unify(a, b, subs):
                        valid = False
                        break

                if valid:

                    remaining = (
                        c1[:i] + c1[i+1:] +
                        c2[:j] + c2[j+1:]
                    )

                    result = [
                        apply_subs(lit, subs)
                        for lit in remaining
                    ]

                    return result

    return None


# Display clauses
def display(clause):

    if not clause:
        return "{}"

    result = []

    for pred, args, neg in clause:

        text = pred + "(" + ", ".join(args) + ")"

        if neg:
            text = "~" + text

        result.append(text)

    return " OR ".join(result)


# STEP 1: Knowledge Base in CNF

# John likes all food
C1 = [
    ("food", ("X",), True),
    ("likes", ("John", "X"), False)
]

# Anything eaten and not killed is food
C2 = [
    ("eats", ("Y", "Z"), True),
    ("killed", ("Y",), False),
    ("food", ("Z",), False)
]

# Anil eats peanuts
C3 = [
    ("eats", ("Anil", "Peanuts"), False)
]

# Anil is alive
C4 = [
    ("alive", ("Anil",), False)
]

# Harry eats everything Anil eats
C5 = [
    ("eats", ("Anil", "V"), True),
    ("eats", ("Harry", "V"), False)
]

# Anyone not killed is alive
C6 = [
    ("killed", ("U",), False),
    ("alive", ("U",), False)
]

# Anyone alive is not killed
C7 = [
    ("alive", ("T",), True),
    ("killed", ("T",), True)
]

# Apple and vegetables are food
C8 = [
    ("food", ("Apple",), False)
]

C9 = [
    ("food", ("Vegetables",), False)
]

# STEP 2: Negate the query
# Query: John likes Peanuts

C10 = [
    ("likes", ("John", "Peanuts"), True)
]


# STEP 3: Display initial clauses

print("INITIAL CLAUSES")

clauses = [C1, C2, C3, C4, C5,
           C6, C7, C8, C9, C10]

for c in clauses:
    print(display(c))


# STEP 4: Apply Resolution

print("\nRESOLUTION STEPS")

# Resolve alive(Anil) with NOT alive(T) OR NOT killed(T)
R1 = resolve(C7, C4)
print("Step 1:", display(R1))

# Resolve eats(Anil, Peanuts) with C2
R2 = resolve(C2, C3)
print("Step 2:", display(R2))

# Resolve killed(Anil) with NOT killed(Anil)
R3 = resolve(R2, R1)
print("Step 3:", display(R3))

# Resolve food(Peanuts) with C1
R4 = resolve(C1, R3)
print("Step 4:", display(R4))

# Resolve likes(John, Peanuts) with negated query
R5 = resolve(R4, C10)
print("Step 5:", display(R5))


# STEP 5: Check the final result

print("\nFINAL RESULT")

if R5 == []:
    print("Empty clause obtained: {}")
    print("Contradiction found!")
    print("John likes Peanuts is PROVED.")
else:
    print("Query could not be proved.")
print("Yaswanth Katasani")

INITIAL CLAUSES
~food(X) OR likes(John, X)
~eats(Y, Z) OR killed(Y) OR food(Z)
eats(Anil, Peanuts)
alive(Anil)
~eats(Anil, V) OR eats(Harry, V)
killed(U) OR alive(U)
~alive(T) OR ~killed(T)
food(Apple)
food(Vegetables)
~likes(John, Peanuts)

RESOLUTION STEPS
Step 1: ~killed(Anil)
Step 2: killed(Anil) OR food(Peanuts)
Step 3: food(Peanuts)
Step 4: likes(John, Peanuts)
Step 5: {}

FINAL RESULT
Empty clause obtained: {}
Contradiction found!
John likes Peanuts is PROVED.
Yaswanth Katasani
